# 第8章: ニューラルネット


In [18]:
# OpenMPライブラリの二重読み込みによるカーネルのクラッシュを防ぐ（torch より前に設定）
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---- import ----
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from gensim.models import KeyedVectors

W2V_PATH = "../GoogleNews-vectors-negative300.bin"   # 学習済み単語ベクトル
DATA_DIR = "../chapter 7/SST-2"                        # 第7章のポジネガ分類データ

## 70. 単語埋め込みの読み込み

事前学習済み単語ベクトルを読み込み、先頭行 (index 0) を `<PAD>` 用の
ゼロベクトルとして予約した単語埋め込み行列を構築する。
あわせて、トークンID と単語の双方向対応 (`word2id` / `id2word`) も保持する。

In [19]:
# 学習済み単語ベクトルの読み込み
kv = KeyedVectors.load_word2vec_format(W2V_PATH, binary=True)

# 学習データに出現し、かつ word2vec に存在する単語で語彙を構築する
# (300万語を全て使うと無駄が多いので、学習データに現れる単語に絞ってメモリを節約する)
def build_vocab(path):
    counter = Counter()
    with open(path, encoding="utf-8") as f:
        next(f)  # ヘッダ sentence\tlabel をスキップ
        for line in f:
            text = line.rstrip("\n").rsplit("\t", 1)[0]
            counter.update(text.split(" "))     # 第7章と同じスペース区切り
    return [w for w in counter if w in kv]       # word2vec に無い単語は除外

vocab = build_vocab(f"{DATA_DIR}/train.tsv")

# トークンID ⇔ 単語 の双方向対応。index 0 は <PAD> に予約する
word2id = {"<PAD>": 0}
id2word = {0: "<PAD>"}
for w in vocab:
    word2id[w] = len(word2id)
    id2word[len(id2word)] = w

# 単語埋め込み行列 (V_w x d_w) を構築する
V_w = len(word2id)      # 語彙数 (<PAD> を含む)
d_w = kv.vector_size    # 次元数
emb = np.zeros((V_w, d_w), dtype=np.float32)   # 先頭行はゼロベクトル (<PAD>) のまま
for w in vocab:
    emb[word2id[w]] = kv[w]                     # 2行目以降に学習済みベクトルを格納

## 71. データセットの読み込み

訓練セット (`train.tsv`) と開発セット (`dev.tsv`) のテキストと極性ラベルを読み込み、
テキストをトークンID列に変換する。単語埋め込みの語彙に無い単語は無視し、
全トークンが語彙外で空になった事例は削除する。
各事例は `{'text', 'label', 'input_ids'}` の辞書で表現する。

In [20]:
def load_dataset(path):
    data = []
    with open(path, encoding="utf-8") as f:
        next(f)  # ヘッダをスキップ
        for line in f:
            text, label = line.rstrip("\n").rsplit("\t", 1)
            # 語彙(word2id)に含まれる単語だけをID化し、語彙外の単語は無視する
            input_ids = [word2id[w] for w in text.split(" ") if w in word2id]
            if not input_ids:        # 空トークン列となった事例は削除
                continue
            data.append({
                "text": text,
                "label": torch.tensor([float(label)]),   # ポジ:1. / ネガ:0.
                "input_ids": torch.tensor(input_ids),
            })
    return data

train = load_dataset(f"{DATA_DIR}/train.tsv")
dev   = load_dataset(f"{DATA_DIR}/dev.tsv")

## 72. Bag of wordsモデルの構築

テキストを単語埋め込みの平均ベクトルで表現し、重みベクトルとの内積で
ポジ／ネガを分類するニューラルネットワーク（ロジスティック回帰）を設計する。

In [21]:
class BoWModel(nn.Module):
    def __init__(self, embeddings, freeze=True):
        super().__init__()
        weight = torch.from_numpy(embeddings)
        # 学習済み埋め込みで初期化。freeze=True なら固定、padding_idx=0 で <PAD> を学習対象外にする
        self.emb = nn.Embedding.from_pretrained(weight, freeze=freeze, padding_idx=0)
        self.fc = nn.Linear(weight.shape[1], 1)   # 重みベクトルとの内積 (d_w -> 1)

    def forward(self, input_ids):
        mask = (input_ids != 0).float()                  # パディング(0)を除外するマスク
        vecs = self.emb(input_ids)                       # (batch, seq, d_w)
        summed = (vecs * mask.unsqueeze(-1)).sum(dim=1)  # パディングを除いた和
        avg = summed / mask.sum(dim=1, keepdim=True).clamp(min=1)   # 平均ベクトル
        return self.fc(avg)                              # logit (batch, 1)

## 73. モデルの学習

単語埋め込み行列を固定したまま（ファインチューニングは行わない）、
モデルの重みベクトルを訓練セット上で学習する。
学習の進捗が分かるよう、エポックごとに平均損失を表示する。

In [22]:
torch.manual_seed(0)
model = BoWModel(emb, freeze=True)          # freeze=True で単語埋め込み行列を固定
criterion = nn.BCEWithLogitsLoss()          # 2値分類の損失（内部でsigmoidを適用）
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for ex in train:                                   # 1事例ずつ学習
        optimizer.zero_grad()
        out = model(ex["input_ids"].unsqueeze(0))      # (1, 1)
        loss = criterion(out, ex["label"].unsqueeze(0))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(train):.4f}")   # 進捗モニタリング

epoch 1: loss = 0.4393
epoch 2: loss = 0.3855
epoch 3: loss = 0.3768
epoch 4: loss = 0.3730
epoch 5: loss = 0.3710
epoch 6: loss = 0.3697
epoch 7: loss = 0.3688
epoch 8: loss = 0.3682
epoch 9: loss = 0.3678
epoch 10: loss = 0.3675


## 74. モデルの評価

学習したモデルの開発セットにおける正解率を求める。

In [23]:
def accuracy(model, data, device="cpu"):
    model.eval()
    correct = 0
    with torch.no_grad():                              # 評価時は勾配を計算しない
        for ex in data:
            out = model(ex["input_ids"].unsqueeze(0).to(device))
            pred = (out > 0).float().cpu().view(-1)    # logit>0 ならポジ(1)と予測
            correct += (pred == ex["label"]).item()
    return correct / len(data)

print(f"開発セット正解率: {accuracy(model, dev):.4f}")

開発セット正解率: 0.7913


## 75. パディング

複数の事例をまとめて一つのテンソルにする関数 `collate` を実装する。
トークン列を最も長いものに揃えて 0番のトークンIDでパディングし、
さらにトークン列が長い順に事例を並び替える。

In [24]:
def collate(batch):
    # トークン列が長い順に並び替える
    batch = sorted(batch, key=lambda ex: len(ex["input_ids"]), reverse=True)
    maxlen = len(batch[0]["input_ids"])               # 最長のトークン列長
    # 0番ID で最長に揃えてパディング
    input_ids = torch.zeros(len(batch), maxlen, dtype=torch.long)
    for i, ex in enumerate(batch):
        input_ids[i, :len(ex["input_ids"])] = ex["input_ids"]
    label = torch.stack([ex["label"] for ex in batch])   # (batch, 1)
    return {"input_ids": input_ids, "label": label}

## 76. ミニバッチ学習

問題75の `collate` を使ってミニバッチで学習し、
学習したモデルの開発セットにおける正解率を求める。

In [25]:
torch.manual_seed(0)
model = BoWModel(emb, freeze=True)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# collate でパディングしながらミニバッチを生成
loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for batch in loader:                              # バッチ単位で学習
        optimizer.zero_grad()
        loss = criterion(model(batch["input_ids"]), batch["label"])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(loader):.4f}")

print(f"開発セット正解率: {accuracy(model, dev):.4f}")

epoch 1: loss = 0.5298
epoch 2: loss = 0.4304
epoch 3: loss = 0.4031
epoch 4: loss = 0.3906
epoch 5: loss = 0.3837
epoch 6: loss = 0.3794
epoch 7: loss = 0.3767
epoch 8: loss = 0.3745
epoch 9: loss = 0.3731
epoch 10: loss = 0.3719
開発セット正解率: 0.7844


## 77. GPU上での学習

問題76のモデル学習を GPU 上で実行し、
学習したモデルの開発セットにおける正解率を求める。

In [26]:
# GPU が使えれば cuda、無ければ cpu を使う
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)
model = BoWModel(emb, freeze=True).to(device)         # モデルを GPU へ転送
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)     # 入力も GPU へ転送
        label = batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(input_ids), label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(loader):.4f}")

print(f"開発セット正解率: {accuracy(model, dev, device):.4f}")

epoch 1: loss = 0.5298
epoch 2: loss = 0.4304
epoch 3: loss = 0.4031
epoch 4: loss = 0.3906
epoch 5: loss = 0.3837
epoch 6: loss = 0.3794
epoch 7: loss = 0.3767
epoch 8: loss = 0.3745
epoch 9: loss = 0.3731
epoch 10: loss = 0.3719
開発セット正解率: 0.7844


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新する
ファインチューニングを導入し、開発セットにおける正解率を求める。

In [27]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)
model = BoWModel(emb, freeze=False).to(device)   # freeze=False で単語埋め込みも更新する
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        label = batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(input_ids), label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(loader):.4f}")

print(f"開発セット正解率: {accuracy(model, dev, device):.4f}")

epoch 1: loss = 0.3864
epoch 2: loss = 0.2400
epoch 3: loss = 0.2000
epoch 4: loss = 0.1797
epoch 5: loss = 0.1675
epoch 6: loss = 0.1593
epoch 7: loss = 0.1547
epoch 8: loss = 0.1504
epoch 9: loss = 0.1473
epoch 10: loss = 0.1448
開発セット正解率: 0.7787


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを変更してモデルを学習し、
開発セットにおける正解率を求める。ここでは平均ベクトルに対して
多層ニューラルネットワーク (MLP) を通す構成にする。

In [28]:
# 平均ベクトル -> 隠れ層(ReLU+Dropout) -> 出力 の多層パーセプトロン
class MLPModel(nn.Module):
    def __init__(self, embeddings, hidden=128, freeze=False):
        super().__init__()
        weight = torch.from_numpy(embeddings)
        self.emb = nn.Embedding.from_pretrained(weight, freeze=freeze, padding_idx=0)
        self.net = nn.Sequential(
            nn.Linear(weight.shape[1], hidden),   # d_w -> hidden
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 1),                 # hidden -> 1
        )

    def forward(self, input_ids):
        mask = (input_ids != 0).float()           # パディングを除外
        vecs = self.emb(input_ids)
        avg = (vecs * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
        return self.net(avg)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
model = MLPModel(emb, hidden=128, freeze=False).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        label = batch["label"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(input_ids), label)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch + 1}: loss = {total_loss / len(loader):.4f}")

print(f"開発セット正解率: {accuracy(model, dev, device):.4f}")

epoch 1: loss = 0.1560
epoch 2: loss = 0.1365
epoch 3: loss = 0.1237
epoch 4: loss = 0.1129
epoch 5: loss = 0.1038
epoch 6: loss = 0.0964
epoch 7: loss = 0.0902
epoch 8: loss = 0.0850
epoch 9: loss = 0.0795
epoch 10: loss = 0.0749
開発セット正解率: 0.7764
